# Solar Farm SCADA — tsqc Quality Control Demo

This notebook runs the full `timeseries-qc` pipeline on one week of synthetic
hourly SCADA data from a single-axis tracking solar farm with three sensor tags:

| Tag | Description | Units | Normal Range |
|-----|-------------|-------|--------------| 
| `INVERTER.MW` | AC output of inverter string array | MW | 0–11 |
| `MET.IRRADIANCE` | Pyranometer global horizontal irradiance (GHI) | W/m² | 0–1050 |
| `TRACKER.ANGLE` | Single-axis tracker tilt angle (E-W) | degrees | -90 to +90 |

**Engineered anomalies** (designed to exercise every built-in rule):

| Anomaly | Tag | Rule triggered | Label |
|---------|-----|----------------|-------|
| Sensor dropout (NaN burst, 5 h) | INVERTER.MW | `NullRule` | bad |
| Delta spike +25 MW in 1 h | INVERTER.MW | `DeltaRule` | sus |
| Pyranometer comm-freeze 10 h | MET.IRRADIANCE | `FlatlineRule` | sus |
| Nighttime irradiance 1100 W/m² (above 1050 limit) | MET.IRRADIANCE | `RangeRule` | bad |
| Tracker angle 110° (beyond physical stops) | TRACKER.ANGLE | `RangeRule` | bad |
| Tracker stuck 15 h at 30° | TRACKER.ANGLE | `FlatlineRule` | sus |
| Duplicate timestamp | TRACKER.ANGLE | `check_timestamps()` | error |

In [1]:
import sys
sys.path.insert(0, '..')  # allow import from repo root when running in examples/

import pandas as pd
import tsqc

print(f'tsqc version: {tsqc.__version__}')

tsqc version: 0.0.1


## 1. Generate / Load the Synthetic Data

In [2]:
import subprocess, os

csv_path = '../data/solar_farm_scada.csv'

if not os.path.exists(csv_path):
    print('Generating synthetic data...')
    subprocess.run([sys.executable, '../data/generate_solar_data.py'], check=True)

df = pd.read_csv(csv_path)
print(f'Loaded {len(df):,} rows × {df["tag_name"].nunique()} tags')
print(f'Tags: {sorted(df["tag_name"].unique())}')
print(f'Date range: {df["timestamp"].min()} → {df["timestamp"].max()}')
df.head(8)

Loaded 505 rows × 3 tags
Tags: ['INVERTER.MW', 'MET.IRRADIANCE', 'TRACKER.ANGLE']
Date range: 2026-06-01T00:00:00+00:00 → 2026-06-07T23:00:00+00:00


,timestamp,tag_name,value
0,2026-06-01T00:00:00+00:00,INVERTER.MW,0.0457
1,2026-06-01T01:00:00+00:00,INVERTER.MW,0.0000
2,2026-06-01T02:00:00+00:00,INVERTER.MW,0.1126
3,2026-06-01T03:00:00+00:00,INVERTER.MW,0.1411
4,2026-06-01T04:00:00+00:00,INVERTER.MW,0.0000
5,2026-06-01T05:00:00+00:00,INVERTER.MW,0.0000
6,2026-06-01T06:00:00+00:00,INVERTER.MW,2.2444
7,2026-06-01T07:00:00+00:00,INVERTER.MW,4.2914


## 2. Run Quality Checks with YAML Rules

In [3]:
result = tsqc.check(
    df,
    rules='../data/solar_rules.yaml',
    assume_tz='UTC',   # CSV has tz-naive ISO strings
)

print(result)
result.df[['timestamp', 'tag_name', 'value', 'quality', 'quality_reasons']].head(12)

QCResult(rows=505, tags=3)


,timestamp,tag_name,value,quality,quality_reasons
0,2026-06-01 00:00:00+00:00,INVERTER.MW,0.0457,good,
1,2026-06-01 01:00:00+00:00,INVERTER.MW,0.0000,sus,flatline
2,2026-06-01 02:00:00+00:00,INVERTER.MW,0.1126,good,
3,2026-06-01 03:00:00+00:00,INVERTER.MW,0.1411,good,
4,2026-06-01 04:00:00+00:00,INVERTER.MW,0.0000,good,
5,2026-06-01 05:00:00+00:00,INVERTER.MW,0.0000,good,
6,2026-06-01 06:00:00+00:00,INVERTER.MW,2.2444,good,
7,2026-06-01 07:00:00+00:00,INVERTER.MW,4.2914,good,
8,2026-06-01 08:00:00+00:00,INVERTER.MW,6.2324,good,
9,2026-06-01 09:00:00+00:00,INVERTER.MW,7.6904,good,


## 3. Quality Summary per Tag

In [4]:
summary = result.summary()
print(summary.to_string(index=False))

      tag_name  total_rows  n_good  n_sus  n_bad  pct_good  pct_sus  pct_bad
   INVERTER.MW         168     132     30      6     78.57    17.86     3.57
 TRACKER.ANGLE         169      97     69      3     57.40    40.83     1.78
MET.IRRADIANCE         168     125     42      1     74.40    25.00     0.60


## 4. Interactive Quality Timeline

One horizontal bar per tag. **Red = bad, Yellow = sus, Green = good.**  
Use the **range slider** at the bottom or the **1d / 1w / All** buttons to zoom.  
Hover any segment to see tag, quality level, start/end time, and duration.

In [5]:
fig = result.plot(
    title='Solar Farm SCADA — June 2026 Quality Timeline',
    height=420,
)
fig.show()

### Zoom: single day with the most anomalies

In [6]:
# Day 3 (2026-06-03) covers: tracker stuck anomaly start + irradiance events
fig_day = result.plot(
    title='Day 3 Detail — 2026-06-03',
    start='2026-06-03',
    end='2026-06-03T23:00:00+00:00',
)
fig_day.show()

## 5. Timestamp Health Check

In [7]:
ts_issues = result.check_timestamps(expected_freq='1h')
print(f'Timestamp issues found: {len(ts_issues)}')
if not ts_issues.empty:
    print(ts_issues.groupby(['tag_name', 'issue_type', 'severity']).size()
          .reset_index(name='count').to_string(index=False))
ts_issues

Timestamp issues found: 1
     tag_name issue_type severity  count
TRACKER.ANGLE  duplicate    error      1


,tag_name,issue_type,timestamp,description,severity
0,TRACKER.ANGLE,duplicate,2026-06-05 04:00:00+00:00,Duplicate timestamp: 2026-06-05 04:00:00+00:00,error


## 6. Investigate Individual Tag Issues

In [8]:
# INVERTER.MW — show all flagged rows and their reasons
inv = result.df[
    (result.df['tag_name'] == 'INVERTER.MW') &
    (result.df['quality'] != 'good')
].copy()

print(f'INVERTER.MW flagged rows: {len(inv)}')
print(inv['quality_reasons'].value_counts().to_string())
inv[['timestamp', 'value', 'quality', 'quality_reasons']].head(20)

INVERTER.MW flagged rows: 36
quality_reasons
flatline       29
null            5
range|delta     1
delta           1


,timestamp,value,quality,quality_reasons
1,2026-06-01 01:00:00+00:00,0.0000,sus,flatline
21,2026-06-01 21:00:00+00:00,0.0000,sus,flatline
25,2026-06-02 01:00:00+00:00,0.0000,sus,flatline
26,2026-06-02 02:00:00+00:00,0.0798,sus,flatline
27,2026-06-02 03:00:00+00:00,0.0548,sus,flatline
28,2026-06-02 04:00:00+00:00,0.0619,sus,flatline
29,2026-06-02 05:00:00+00:00,0.0646,sus,flatline
45,2026-06-02 21:00:00+00:00,0.0328,sus,flatline
48,2026-06-03 00:00:00+00:00,0.1018,sus,flatline
49,2026-06-03 01:00:00+00:00,0.0101,sus,flatline


In [9]:
# MET.IRRADIANCE — comm-freeze flatline and nighttime spike
irr = result.df[
    (result.df['tag_name'] == 'MET.IRRADIANCE') &
    (result.df['quality'] != 'good')
].copy()

print(f'MET.IRRADIANCE flagged rows: {len(irr)}')
print(irr['quality_reasons'].value_counts().to_string())
irr[['timestamp', 'value', 'quality', 'quality_reasons']]

MET.IRRADIANCE flagged rows: 43
quality_reasons
flatline       40
delta           2
range|delta     1


,timestamp,value,quality,quality_reasons
169,2026-06-01 01:00:00+00:00,0.00,sus,flatline
170,2026-06-01 02:00:00+00:00,0.00,sus,flatline
173,2026-06-01 05:00:00+00:00,0.00,sus,flatline
188,2026-06-01 20:00:00+00:00,0.00,sus,flatline
190,2026-06-01 22:00:00+00:00,20.68,sus,flatline
192,2026-06-02 00:00:00+00:00,0.00,sus,flatline
195,2026-06-02 03:00:00+00:00,0.00,sus,flatline
198,2026-06-02 06:00:00+00:00,0.00,sus,flatline
199,2026-06-02 07:00:00+00:00,0.00,sus,flatline
200,2026-06-02 08:00:00+00:00,0.00,sus,flatline


In [10]:
# TRACKER.ANGLE — out-of-range and stuck period
trk = result.df[
    (result.df['tag_name'] == 'TRACKER.ANGLE') &
    (result.df['quality'] != 'good')
].copy()

print(f'TRACKER.ANGLE flagged rows: {len(trk)}')
print(trk['quality_reasons'].value_counts().to_string())

# The stuck tracker at 30° — 15 consecutive hours
stuck = trk[trk['quality_reasons'].str.contains('flatline', na=False)]
print(f'\nTracker stuck (flatline) period:')
print(f'  Start: {stuck["timestamp"].min()}')
print(f'  End:   {stuck["timestamp"].max()}')
print(f'  Duration: {len(stuck)} hours')

TRACKER.ANGLE flagged rows: 72
quality_reasons
flatline       55
delta          14
range           2
range|delta     1

Tracker stuck (flatline) period:
  Start: 2026-06-01 01:00:00+00:00
  End:   2026-06-07 23:00:00+00:00
  Duration: 55 hours


## 7. Export Self-Contained HTML Report

In [11]:
report_path = '../data/solar_qc_report.html'
result.export_report(
    report_path,
    title='Solar Farm SCADA — June 2026 Quality Report'
)
import os
print(f'Report written to: {report_path}')
print(f'File size: {os.path.getsize(report_path)/1024:.0f} KB')

Report written to: ../data/solar_qc_report.html
File size: 3645 KB


---

## Summary of QC Results

After running `tsqc.check()` with `solar_rules.yaml`:

- **INVERTER.MW**: NaN burst (5 bad rows) + delta spike of +25 MW (1 sus row)
- **MET.IRRADIANCE**: Comm-freeze flatline (10 sus rows) + nighttime 980 W/m² reading (1 bad row)
- **TRACKER.ANGLE**: Out-of-range 110° (3 bad rows) + tracker stuck 15 h (sus rows) + 1 duplicate timestamp

**Key YAML tuning insights:**
- `min_delta: 1.0` on `MET.IRRADIANCE` and `TRACKER.ANGLE` flatline rules prevents false-positive
  flags during legitimate nighttime periods where the sensor correctly reads a constant 0/flat value.
- `min_delta: 0.05` on `INVERTER.MW` similarly allows the inverter to sit at ~0 MW at night without triggering flatline.
- The `delta` threshold of `5.0 MW/h` for the inverter catches the engineered +25 MW spike  
  while ignoring normal ramp rates during morning startup (~2–3 MW/h).